# TUM-VIE: DEVO-Style Recording Transfer Benchmark

## Research Question
Can a spiking neural network trained on one TUM-VIE recording generalise to an entirely
**different recording — without any fine-tuning?**

| Split | Recording | Motion Profile | Environment |
|-------|-----------|---------------|-------------|
| **Train** | `mocap-6dof` | Controlled 6-DoF handheld | Indoor lab |
| **Test** | `running-easy` | Running gait, faster dynamics | Outdoor |

No train/test data overlap — this is **pure cross-domain transfer** (analogous to DEVO-style generalisation).

---

## Experimental Protocol
1. Train each model exclusively on `mocap-6dof` frames
2. Evaluate zero-shot on `running-easy`
3. Report **6-DoF pose MAE** (translation m, rotation °) per axis + aggregate RMSE
4. Track **spike rate** per epoch (spikes/neuron/timestep) as energy proxy (lower = more efficient)
5. Measure **wall-clock latency** (JIT-warmed, 50-run mean) and **parameter count**

## Baselines (from weakest to strongest)
| Model | Modality | Description |
|-------|----------|-------------|
| **ZeroVelocity** | — | Predict training-set mean; absolute lower bound |
| **IMU-MLP** | IMU only | sklearn 2-layer MLP on aggregated 6-axis IMU |
| **ConvLIF-SNN** | Events | Spiking Conv encoder, vision only |
| **IMUConditioned-SNN** | Events + IMU | Late-gated visual–IMU fusion |

---
*Results are saved to `experiments/quant_pipeline/pipeline_runs.jsonl` for longitudinal tracking.*

In [ ]:
from __future__ import annotations
from pathlib import Path
import time, json, datetime
from collections import defaultdict

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optax
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from tonic import transforms
from tonic.datasets import TUMVIE

import spyx.models as fm

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

# ── Experiment hyper-parameters ───────────────────────────────────────────────
SEED            = 123
TRAIN_REC       = "mocap-6dof"
TEST_REC        = "running-easy"
SPATIAL_FACTOR  = 0.25       # spatial downscale to reduce memory
N_TIME_BINS     = 24         # temporal bins per sample window
TRAIN_SAMPLES   = 96         # increase to ≥512 with full data
TEST_SAMPLES    = 48
BATCH_SIZE      = 8
EPOCHS          = 20
LR              = 2e-3

# ── I/O paths ─────────────────────────────────────────────────────────────────
data_root    = Path("../data").resolve()
data_root.mkdir(parents=True, exist_ok=True)
FIG_DIR      = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)
RESULTS_FILE = Path("../../experiments/quant_pipeline/pipeline_runs.jsonl").resolve()

# ── Colour palettes ──────────────────────────────────────────────────────────
AXIS_LABELS  = ["tx (m)", "ty (m)", "tz (m)", "rx (°)", "ry (°)", "rz (°)"]
MODEL_COLORS = {
    "ZeroVelocity":        "#aaaaaa",
    "IMU-MLP":             "#fc8d62",
    "ConvLIF-SNN":         "#8da0cb",
    "IMUConditioned-SNN":  "#66c2a5",
}

# ── Tonic dataset + transform ─────────────────────────────────────────────────
sensor       = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR),
                int(sensor[1] * SPATIAL_FACTOR),
                sensor[2])
tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_TIME_BINS),
])
print(f"Sensor (original): {sensor}  →  (downsampled): {sensor_small}")

/Users/vincent/Work/autoresearch-mlx/.bench-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Data Loading & Preprocessing
# ─────────────────────────────────────────────────────────────────────────────

def quat_to_euler_xyz(qx: float, qy: float, qz: float, qw: float) -> np.ndarray:
    """Unit quaternion → Euler angles (XYZ intrinsic, radians)."""
    t0 = 2.0 * (qw * qx + qy * qz)
    t1 = 1.0 - 2.0 * (qx**2 + qy**2)
    rx = np.arctan2(t0, t1)
    t2 = np.clip(2.0 * (qw * qy - qz * qx), -1.0, 1.0)
    ry = np.arcsin(t2)
    t3 = 2.0 * (qw * qz + qx * qy)
    t4 = 1.0 - 2.0 * (qy**2 + qz**2)
    rz = np.arctan2(t3, t4)
    return np.array([rx, ry, rz], dtype=np.float32)


def load_recording(recording: str, limit: int):
    """Load `limit` samples from one TUMVIE recording.
    Returns:
        X: (N, T, H, W, C) event frames  — batch-major
        U: (N, 6)           mean IMU (acc xyz, gyr xyz)
        Y: (N, 6)           pose (translation m, rotation rad)
    """
    ds = TUMVIE(save_to=str(data_root), recording=recording, transform=tfm)
    xs, us, ys = [], [], []
    n = min(limit, len(ds))
    for i in range(n):
        d, t = ds[i]
        left  = np.asarray(d["events_left"],  dtype=np.float32)  # (T, H, W, P)
        right = np.asarray(d["events_right"], dtype=np.float32)  # (T, H, W, P)
        # Stack stereo along the height axis → (T, 2H, W, P)
        x = np.concatenate([left, right], axis=1)

        imu = np.asarray(d["imu"], dtype=np.float32)
        if imu.ndim == 2 and imu.shape[0] > 0:
            imu = imu.mean(axis=0)          # average over time window → (6,)
        else:
            imu = np.zeros(6, dtype=np.float32)

        mocap = np.asarray(t["mocap"], dtype=np.float32)
        mocap = mocap[-1] if mocap.ndim == 2 else mocap
        trans = mocap[:3]                               # raw translation (m)
        rot   = quat_to_euler_xyz(*mocap[3:7])          # quaternion → Euler rad
        y = np.concatenate([trans, rot]).astype(np.float32)

        xs.append(x); us.append(imu); ys.append(y)

    return np.stack(xs), np.stack(us), np.stack(ys)  # (N, T, H, W, C), (N,6), (N,6)


print("▶ Loading train recording:", TRAIN_REC)
Xtr_np, Utr_np, Ytr_np = load_recording(TRAIN_REC, TRAIN_SAMPLES)
print("▶ Loading test recording :", TEST_REC)
Xte_np, Ute_np, Yte_np = load_recording(TEST_REC, TEST_SAMPLES)

# ── Shape inspection ──────────────────────────────────────────────────────────
N_TRAIN, T_BINS, H_SPATIAL, W_SPATIAL, N_CHANNELS = Xtr_np.shape
INPUT_HW       = (H_SPATIAL, W_SPATIAL)
INPUT_CHANNELS = N_CHANNELS
IMU_DIM        = Utr_np.shape[-1]   # 6
OUTPUT_DIM     = Ytr_np.shape[-1]   # 6

print(f"\nTrain  X{Xtr_np.shape}  U{Utr_np.shape}  Y{Ytr_np.shape}")
print(f"Test   X{Xte_np.shape}  U{Ute_np.shape}  Y{Yte_np.shape}")
print(f"Model: input_hw={INPUT_HW}, channels={INPUT_CHANNELS}, T={T_BINS}")

# ── Target normalisation (z-score; rotation converted to degrees) ─────────────
# Using degrees for rotation makes per-axis MAE directly interpretable.
def deg_targets(Y_rad: np.ndarray) -> np.ndarray:
    Y = Y_rad.copy(); Y[:, 3:] = np.degrees(Y[:, 3:]); return Y

Ytr_deg = deg_targets(Ytr_np)
Yte_deg = deg_targets(Yte_np)

Y_mean = Ytr_deg.mean(axis=0, keepdims=True)          # (1, 6) training statistics
Y_std  = Ytr_deg.std(axis=0,  keepdims=True) + 1e-6

Ytr_z  = ((Ytr_deg - Y_mean) / Y_std).astype(np.float32)
Yte_z  = ((Yte_deg - Y_mean) / Y_std).astype(np.float32)

# ── JAX arrays (keep batch-major; transpose to time-major inside model calls) ─
Xtr_j  = jnp.asarray(Xtr_np)
Utr_j  = jnp.asarray(Utr_np)
Ytr_j  = jnp.asarray(Ytr_z)
Xte_j  = jnp.asarray(Xte_np)
Ute_j  = jnp.asarray(Ute_np)
Yte_j  = jnp.asarray(Yte_z)

# ── Helper: batching with time-major transpose for the SNN models ─────────────
def make_batch(X, U, Y, bs, key):
    """Sample a random batch; returns time-major event tensor (T, B, H, W, C)."""
    idx = jax.random.choice(key, X.shape[0], shape=(bs,), replace=False)
    xb  = jnp.swapaxes(X[idx], 0, 1)   # (T, B, H, W, C)
    return xb, U[idx], Y[idx]

def eval_batches(X, U, Y):
    """Iterate over the dataset in non-overlapping batch-sized windows."""
    n = (X.shape[0] // BATCH_SIZE) * BATCH_SIZE   # drop remainder
    for b in range(0, n, BATCH_SIZE):
        xb = jnp.swapaxes(X[b:b+BATCH_SIZE], 0, 1)   # (T, B, H, W, C)
        yield xb, U[b:b+BATCH_SIZE], Y[b:b+BATCH_SIZE]

# ── Visualise target distributions (train vs test shift) ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(11, 4))
for i, ax in enumerate(axes.flat):
    ax.hist(Ytr_deg[:, i], bins=20, alpha=0.7, color="#377eb8", label="train", density=True)
    ax.hist(Yte_deg[:, i], bins=20, alpha=0.7, color="#e41a1c", label="test",  density=True)
    ax.set_title(AXIS_LABELS[i]); ax.set_xlabel("value (m or °)")
    if i == 0: ax.legend(fontsize=7)
fig.suptitle(f"Target Distribution Shift: {TRAIN_REC} → {TEST_REC}", fontweight="bold", y=1.01)
fig.tight_layout()
plt.savefig(FIG_DIR / "transfer_target_distribution.png", bbox_inches="tight")
plt.show()
print(f"\nY_mean (degrees): { {l: round(float(v),3) for l,v in zip(AXIS_LABELS, Y_mean[0])} }")
print(f"Y_std  (degrees): { {l: round(float(v),3) for l,v in zip(AXIS_LABELS, Y_std[0])} }")

1415552000it [00:54, 26056771.76it/s]                                


1336372224it [01:10, 18946687.23it/s]                                


313885696it [00:20, 15387026.79it/s]                               


Extracting /Users/vincent/Work/autoresearch-mlx/spyx/research/data/TUMVIE/mocap-1d-trans/mocap-1d-trans-vi_gt_data.tar.gz to /Users/vincent/Work/autoresearch-mlx/spyx/research/data/TUMVIE/mocap-1d-trans


1913986048it [01:38, 19491905.91it/s]                                


...

17868884992it [16:27, 18090570.06it/s]                                 


17858608128it [16:54, 17610913.59it/s]                                 


2732484608it [02:27, 18488269.76it/s]                                


Extracting /Users/vincent/Work/autoresearch-mlx/spyx/research/data/TUMVIE/loop-floor0/loop-floor0-vi_gt_data.tar.gz to /Users/vincent/Work/autoresearch-mlx/spyx/research/data/TUMVIE/loop-floor0


 60%|█████▉    | 9604368384/16041505512 [08:47<04:53, 21914860.82it/s] 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.  Non-spiking Baselines
# ─────────────────────────────────────────────────────────────────────────────

# ── Baseline A: Zero-Velocity Predictor ──────────────────────────────────────
# Predicts the training-set mean for every test sample.
# Any model that can't beat this has learned nothing useful.
zero_pred_deg  = np.broadcast_to(Y_mean, Yte_deg.shape)
zero_mae_axis  = np.mean(np.abs(zero_pred_deg - Yte_deg), axis=0)
zero_rmse      = float(np.sqrt(np.mean(zero_mae_axis ** 2)))
print(f"[ZeroVelocity] test RMSE={zero_rmse:.4f} | per-axis MAE: {zero_mae_axis.round(3).tolist()}")

# ── Baseline B: IMU-only sklearn MLP ─────────────────────────────────────────
# Demonstrates how much of the signal is recoverable from IMU alone.
imu_scaler = StandardScaler()
Utr_scaled = imu_scaler.fit_transform(Utr_np)
Ute_scaled = imu_scaler.transform(Ute_np)

mlp_imu = MLPRegressor(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    max_iter=1000,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.15,
)
mlp_imu.fit(Utr_scaled, Ytr_z)    # fit on z-scored targets
imu_pred_z   = mlp_imu.predict(Ute_scaled)
imu_pred_deg = imu_pred_z * Y_std + Y_mean    # denormalise
imu_mae_axis = np.mean(np.abs(imu_pred_deg - Yte_deg), axis=0)
imu_rmse     = float(np.sqrt(np.mean(imu_mae_axis ** 2)))
print(f"[IMU-MLP]       test RMSE={imu_rmse:.4f} | per-axis MAE: {imu_mae_axis.round(3).tolist()}")

# Collect baseline results for later comparison
baseline_results = {
    "ZeroVelocity": {
        "mae_per_axis": zero_mae_axis.tolist(),
        "test_rmse":    zero_rmse,
        "spike_rate":   0.0,
        "n_params":     0,
        "latency_ms":   0.0,
    },
    "IMU-MLP": {
        "mae_per_axis": imu_mae_axis.tolist(),
        "test_rmse":    imu_rmse,
        "spike_rate":   0.0,
        "n_params":     int(sum(w.size for w in mlp_imu.coefs_ + mlp_imu.intercepts_)),
        "latency_ms":   0.0,
    },
}

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.  SNN Model Definitions  (correct ConvConfig API)
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: ConvLIFSNN expects time-major input: (T, batch, H, W, C)
# NOTE: IMUConditioned imu branch averages MLP outputs over axis=0 (batch dim);
#       this means imu gating is identical across the batch — a known limitation
#       that still provides a population-level correction signal.

def conv_lif_forward(x_seq):
    """Pure visual pathway: Spiking ConvNet → 6-DoF."""
    cfg = fm.ConvConfig(
        input_hw=INPUT_HW,
        input_channels=INPUT_CHANNELS,
        channels1=32,
        channels2=64,
        output_dim=OUTPUT_DIM,
    )
    return fm.ConvLIFSNN(cfg)(x_seq)   # → (batch, 6), {"logits_seq", "spike_rate"}


def imu_conditioned_forward(x_seq, imu):
    """Late-gated visual + IMU fusion: IMU branch shifts the visual readout."""
    vis_cfg = fm.ConvConfig(
        input_hw=INPUT_HW,
        input_channels=INPUT_CHANNELS,
        channels1=32,
        channels2=64,
        output_dim=OUTPUT_DIM,
    )
    cfg = fm.IMUConditionedConfig(
        vision_cfg=vis_cfg,
        imu_dim=IMU_DIM,
        imu_hidden=128,
    )
    return fm.IMUConditionedVisualSNN(cfg)(x_seq, imu)   # imu: (batch, 6)


# ── registry: name → (forward_fn, uses_imu, display_name) ────────────────────
MODEL_REGISTRY = {
    "ConvLIF-SNN":         (conv_lif_forward,      False),
    "IMUConditioned-SNN":  (imu_conditioned_forward, True),
}


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Training Infrastructure
# ─────────────────────────────────────────────────────────────────────────────

def count_params(params) -> int:
    return int(sum(x.size for x in jax.tree_util.tree_leaves(params)))


def train_model(display_name: str, forward_fn, use_imu: bool) -> dict:
    """Full training loop with per-epoch history + final evaluation."""

    # ── Initialise ────────────────────────────────────────────────────────────
    key0 = jax.random.PRNGKey(SEED)
    init_xb, init_ub, _ = make_batch(Xtr_j, Utr_j, Ytr_j, BATCH_SIZE, key0)

    if use_imu:
        transformed = hk.without_apply_rng(hk.transform(
            lambda x, u: forward_fn(x, u)))
        params = transformed.init(key0, init_xb, init_ub)
    else:
        transformed = hk.without_apply_rng(hk.transform(
            lambda x: forward_fn(x)))
        params = transformed.init(key0, init_xb)

    n_params  = count_params(params)
    opt       = optax.adamw(LR, weight_decay=1e-4)
    opt_state = opt.init(params)

    # ── JIT-compiled step functions ───────────────────────────────────────────
    if use_imu:
        @jax.jit
        def train_step(params, opt_state, xb, ub, yb):
            def loss_fn(p):
                pred, aux = transformed.apply(p, xb, ub)
                mse = jnp.mean((pred - yb) ** 2)
                return mse, (pred, aux)
            (loss, (pred, aux)), grads = jax.value_and_grad(
                loss_fn, has_aux=True)(params)
            updates, new_opt = opt.update(grads, opt_state, params)
            new_params = optax.apply_updates(params, updates)
            return new_params, new_opt, loss, pred, aux

        @jax.jit
        def infer(params, xb, ub):
            return transformed.apply(params, xb, ub)
    else:
        @jax.jit
        def train_step(params, opt_state, xb, ub, yb):
            def loss_fn(p):
                pred, aux = transformed.apply(p, xb)
                mse = jnp.mean((pred - yb) ** 2)
                return mse, (pred, aux)
            (loss, (pred, aux)), grads = jax.value_and_grad(
                loss_fn, has_aux=True)(params)
            updates, new_opt = opt.update(grads, opt_state, params)
            new_params = optax.apply_updates(params, updates)
            return new_params, new_opt, loss, pred, aux

        @jax.jit
        def infer(params, xb, ub):
            return transformed.apply(params, xb)

    # ── Per-epoch training ────────────────────────────────────────────────────
    history     = defaultdict(list)
    key         = jax.random.PRNGKey(SEED + 7)
    preds_store = []   # running predictions for last epoch

    for epoch in range(1, EPOCHS + 1):
        key, sk = jax.random.split(key)
        xb, ub, yb = make_batch(Xtr_j, Utr_j, Ytr_j, BATCH_SIZE, sk)
        params, opt_state, tr_loss, _, tr_aux = train_step(
            params, opt_state, xb, ub, yb)

        # ── Eval on full test set in batches ──────────────────────────────────
        te_preds, te_maes = [], []
        for xte_b, ute_b, yte_b in eval_batches(Xte_j, Ute_j, Yte_j):
            pred_b, _ = infer(params, xte_b, ute_b)
            mae_b     = np.mean(np.abs(np.asarray(pred_b) - np.asarray(yte_b)), axis=0)
            te_preds.append(np.asarray(pred_b))
            te_maes.append(mae_b)

        te_mae_axis = np.mean(te_maes, axis=0)               # (6,)
        te_rmse     = float(np.sqrt(np.mean(te_mae_axis**2)))
        sr          = float(np.mean(np.asarray(
            tr_aux.get("spike_rate", jnp.zeros(1)))))

        history["epoch"].append(epoch)
        history["train_mse"].append(float(tr_loss))
        history["test_rmse"].append(te_rmse)
        history["test_mae_per_axis"].append(te_mae_axis.tolist())
        history["spike_rate"].append(sr)
        preds_store = te_preds   # overwrite each epoch — keep final epoch

        if epoch % 5 == 0 or epoch == 1:
            print(f"  [{display_name}] ep {epoch:3d} | "
                  f"train_mse={float(tr_loss):.4f}  "
                  f"test_rmse={te_rmse:.4f}  "
                  f"spike_rate={sr:.3f}")

    # ── Final predictions (denormalised) ─────────────────────────────────────
    preds_z   = np.concatenate(preds_store, axis=0)
    preds_deg = preds_z * Y_std + Y_mean
    n_eval    = preds_deg.shape[0]
    final_mae = np.mean(np.abs(preds_deg - Yte_deg[:n_eval]), axis=0)

    # ── Latency benchmark (50 warm runs, single batch) ────────────────────────
    _xb, _ub, _ = make_batch(Xtr_j, Utr_j, Ytr_j, BATCH_SIZE,
                              jax.random.PRNGKey(0))
    infer(params, _xb, _ub)   # JIT warmup
    t0 = time.perf_counter()
    for _ in range(50):
        jax.block_until_ready(infer(params, _xb, _ub))
    latency_ms = (time.perf_counter() - t0) / 50 * 1000

    return {
        "name":               display_name,
        "history":            dict(history),
        "final_mae_per_axis": final_mae.tolist(),
        "test_rmse":          float(np.sqrt(np.mean(final_mae**2))),
        "spike_rate":         history["spike_rate"][-1],
        "n_params":           n_params,
        "latency_ms":         latency_ms,
        "preds_deg":          preds_deg,
        "n_eval":             n_eval,
    }

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  Run All SNN Models
# ─────────────────────────────────────────────────────────────────────────────
snn_results = {}
for name, (fwd_fn, use_imu) in MODEL_REGISTRY.items():
    print(f"\n{'='*60}\n  Training: {name}\n{'='*60}")
    snn_results[name] = train_model(name, fwd_fn, use_imu)

# Merge all results for visualisations
all_results = {**baseline_results}
for name, r in snn_results.items():
    all_results[name] = {
        "mae_per_axis": r["final_mae_per_axis"],
        "test_rmse":    r["test_rmse"],
        "spike_rate":   r["spike_rate"],
        "n_params":     r["n_params"],
        "latency_ms":   r["latency_ms"],
    }

print("\n\n── Final Test RMSE (lower is better) ──────────────────────────────")
for name, res in all_results.items():
    stars = "★" if res["test_rmse"] == min(r["test_rmse"] for r in all_results.values()) else " "
    print(f"  {stars} {name:<28}  RMSE={res['test_rmse']:.4f}  "
          f"params={res.get('n_params', 0):,}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 1 – Training Curves
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for name, res in snn_results.items():
    h     = res["history"]
    color = MODEL_COLORS.get(name, "#333333")
    epochs_x = h["epoch"]
    axes[0].plot(epochs_x, h["train_mse"],   label=name, color=color, lw=1.8)
    axes[1].plot(epochs_x, h["test_rmse"],   label=name, color=color, lw=1.8)
    axes[2].plot(epochs_x, h["spike_rate"],  label=name, color=color, lw=1.8)

for ax, title, ylabel in zip(axes,
        ["Train MSE (z-scored)", "Test RMSE (z-scored)", "Mean Spike Rate"],
        ["MSE", "RMSE", "spikes / neuron / step"]):
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

fig.suptitle(f"Learning Dynamics — {TRAIN_REC} → {TEST_REC}", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "transfer_learning_curves.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 2 – Per-Axis MAE Heatmap
# ─────────────────────────────────────────────────────────────────────────────
model_names_ordered = list(all_results.keys())
mae_matrix = np.array([all_results[n]["mae_per_axis"] for n in model_names_ordered])

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [2, 1]})

# Heatmap
im = axes[0].imshow(mae_matrix, aspect="auto",
                    cmap="YlOrRd", vmin=0, vmax=mae_matrix.max())
axes[0].set_xticks(range(len(AXIS_LABELS)));  axes[0].set_xticklabels(AXIS_LABELS, rotation=30, ha="right")
axes[0].set_yticks(range(len(model_names_ordered))); axes[0].set_yticklabels(model_names_ordered)
axes[0].set_title("Per-Axis MAE Heatmap (lower = better)", fontweight="bold")
plt.colorbar(im, ax=axes[0], label="MAE (m or °)")
for i in range(len(model_names_ordered)):
    for j in range(len(AXIS_LABELS)):
        axes[0].text(j, i, f"{mae_matrix[i,j]:.2f}", ha="center", va="center",
                     fontsize=8, color="white" if mae_matrix[i,j] > mae_matrix.max()*0.6 else "black")

# Grouped bar chart (aggregate RMSE)
rmses  = [all_results[n]["test_rmse"] for n in model_names_ordered]
colors = [MODEL_COLORS.get(n, "#888888") for n in model_names_ordered]
bars   = axes[1].barh(model_names_ordered[::-1], rmses[::-1],
                       color=colors[::-1], edgecolor="white", height=0.6)
axes[1].set_xlabel("Test RMSE (z-scored units)")
axes[1].set_title("Overall RMSE", fontweight="bold")
for bar, v in zip(bars, rmses[::-1]):
    axes[1].text(v + 0.005, bar.get_y() + bar.get_height()/2,
                 f"{v:.3f}", va="center", fontsize=8)

fig.suptitle(f"Cross-Recording Error Analysis: {TRAIN_REC} → {TEST_REC}", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "transfer_mae_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 3 – Predicted vs Ground-Truth Trajectory (3D translation)
# ─────────────────────────────────────────────────────────────────────────────
gt_deg = Yte_deg   # (N, 6) ground-truth in metres/degrees

fig = plt.figure(figsize=(15, 9))
fig.suptitle(f"Predicted Trajectories: {TEST_REC}  (translation xyz only)",
             fontweight="bold", fontsize=11)

rows = len(snn_results) + 1          # GT + one row per SNN model
gs   = gridspec.GridSpec(rows, 3, figure=fig, hspace=0.45)

def _plot_traj_row(row_idx, label, preds, alpha=1.0, is_gt=False):
    subset = min(preds.shape[0], gt_deg.shape[0])
    for col, (ax_label, cidx) in enumerate(zip(["tx (m)", "ty (m)", "tz (m)"], [0, 1, 2])):
        ax = fig.add_subplot(gs[row_idx, col])
        ax.plot(gt_deg[:subset, cidx], color="#555555", lw=1, label="GT", alpha=0.8)
        if not is_gt:
            ax.plot(preds[:subset, cidx], color=MODEL_COLORS.get(label, "#e41a1c"),
                    lw=1.2, label=label, alpha=alpha)
        ax.set_title(ax_label if row_idx == 0 else "", fontsize=8)
        ax.set_ylabel(label if col == 0 else "", fontsize=8)
        ax.set_xlabel("sample index" if row_idx == rows - 1 else "")
        if col == 0 and row_idx == 1:
            ax.legend(fontsize=7, loc="upper right")

for r, (name, res) in enumerate(snn_results.items()):
    n_eval = res["n_eval"]
    _plot_traj_row(r, name, res["preds_deg"])

plt.savefig(FIG_DIR / "transfer_trajectory_prediction.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 4 – Efficiency Frontier: Spike Rate vs Accuracy
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Spike rate vs RMSE  (only SNN models have spike_rate)
for name, res in snn_results.items():
    axes[0].scatter(
        res["spike_rate"],  res["test_rmse"],
        s=100, label=name,
        color=MODEL_COLORS.get(name, "#888888"),
        edgecolors="white", linewidths=1.5, zorder=3,
    )
    axes[0].annotate(name, (res["spike_rate"], res["test_rmse"]),
                     textcoords="offset points", xytext=(6, 0), fontsize=7)

axes[0].set_xlabel("Mean Spike Rate (spikes/neuron/step)")
axes[0].set_ylabel("Test RMSE (z-scored)")
axes[0].set_title("Energy Proxy vs Accuracy Trade-off", fontweight="bold")
axes[0].set_xlim(left=0)

# Params vs RMSE (all models)
for name, res in all_results.items():
    n = max(res.get("n_params", 0), 1)
    axes[1].scatter(
        np.log10(n), res["test_rmse"],
        s=100,  label=name,
        color=MODEL_COLORS.get(name, "#888888"),
        edgecolors="white", linewidths=1.5, zorder=3,
    )
    axes[1].annotate(name, (np.log10(n), res["test_rmse"]),
                     textcoords="offset points", xytext=(4, 0), fontsize=7)

axes[1].set_xlabel("log₁₀(Parameter Count)")
axes[1].set_ylabel("Test RMSE (z-scored)")
axes[1].set_title("Model Capacity vs Accuracy", fontweight="bold")

for ax in axes:
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Efficiency Analysis", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "transfer_efficiency_frontier.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Fig 5 – Residual Error Distribution  (box-plots per axis)
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for ax_idx, ax in enumerate(axes.flat):
    ax.axhline(0, color="#444444", lw=0.7, ls="--")
    for m_idx, (name, res) in enumerate(snn_results.items()):
        n_eval = res["n_eval"]
        residuals = res["preds_deg"][:, ax_idx] - Yte_deg[:n_eval, ax_idx]
        ax.boxplot(residuals,
                   positions=[m_idx],
                   widths=0.5,
                   patch_artist=True,
                   boxprops=dict(facecolor=MODEL_COLORS.get(name, "#88c"), alpha=0.7),
                   medianprops=dict(color="white", lw=2),
                   flierprops=dict(marker=".", markersize=3, color="#666"),
                   whiskerprops=dict(color="#333"),
                   capprops=dict(color="#333"),
                   )
    ax.set_xticks(range(len(snn_results)))
    ax.set_xticklabels(list(snn_results.keys()), rotation=15, ha="right", fontsize=7)
    ax.set_title(AXIS_LABELS[ax_idx], fontweight="bold")
    ax.set_ylabel("Residual (m or °)")

fig.suptitle("Residual Error Distributions per Axis (test set)", fontweight="bold")
fig.tight_layout()
plt.savefig(FIG_DIR / "transfer_residual_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Summary Table + Save Results to pipeline_runs.jsonl
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for name, res in all_results.items():
    mae = res["mae_per_axis"]
    row = {
        "model":        name,
        "t_mae_m":      round(float(np.mean(mae[:3])), 4),   # translation MAE (m)
        "r_mae_deg":    round(float(np.mean(mae[3:])), 4),   # rotation MAE (°)
        "test_rmse":    round(res["test_rmse"], 4),
        "spike_rate":   round(res.get("spike_rate", 0.0), 4),
        "n_params":     res.get("n_params", 0),
        "latency_ms":   round(res.get("latency_ms", 0.0), 2),
    }
    rows.append(row)

df = pd.DataFrame(rows).set_index("model").sort_values("test_rmse")
print("\n── Cross-Recording Transfer Benchmark ─────────────────────────────────")
print(df.to_string())

# ── Persist to JSONL ──────────────────────────────────────────────────────────
run_record = {
    "timestamp":    datetime.datetime.utcnow().isoformat() + "Z",
    "experiment":   "tumvie_recording_transfer",
    "train_rec":    TRAIN_REC,
    "test_rec":     TEST_REC,
    "epochs":       EPOCHS,
    "train_n":      TRAIN_SAMPLES,
    "test_n":       TEST_SAMPLES,
    "results":      {
        name: {k: v for k, v in res.items() if k != "preds_deg"}
        for name, res in all_results.items()
    },
    "snn_history":  {name: r["history"] for name, r in snn_results.items()},
}
RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_FILE, "a") as f:
    f.write(json.dumps(run_record) + "\n")
print(f"\n✓ Results appended to {RESULTS_FILE}")